In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [17]:
df = pd.read_csv(r"E:\Program Files\Call Center.csv")  
df.head()

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,call_center
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,10/29/2020,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles/CA
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,NaN,10/05/2020,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore/MD
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,NaN,10/04/2020,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles/CA
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,10/17/2020,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles/CA
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,NaN,10/17/2020,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles/CA


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32941 entries, 0 to 32940
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        32941 non-null  object 
 1   customer_name             32941 non-null  object 
 2   sentiment                 32941 non-null  object 
 3   csat_score                12271 non-null  float64
 4   call_timestamp            32941 non-null  object 
 5   reason                    32941 non-null  object 
 6   city                      32941 non-null  object 
 7   state                     32941 non-null  object 
 8   channel                   32941 non-null  object 
 9   response_time             32941 non-null  object 
 10  call duration in minutes  32941 non-null  int64  
 11  call_center               32941 non-null  object 
dtypes: float64(1), int64(1), object(10)
memory usage: 3.0+ MB


In [19]:
df.isnull().sum()

id                              0
customer_name                   0
sentiment                       0
csat_score                  20670
call_timestamp                  0
reason                          0
city                            0
state                           0
channel                         0
response_time                   0
call duration in minutes        0
call_center                     0
dtype: int64

In [45]:
print(f"Total rows: {len(df)}")

Total rows: 32941


In [22]:
# 1. Fill missing CSAT values with a string placeholder or a safe flag
# We use a string here, but ensure the column type can handle mixed/object types if needed, 
# or convert the numeric scores to strings for uniform categorical reporting.
df['csat_score_clean'] = df['csat_score'].fillna('No Survey')

# 2. Check the value counts to make sure it worked perfectly
print("--- CSAT Distribution (Including Missing Data) ---")
print(df['csat_score_clean'].value_counts())

--- CSAT Distribution (Including Missing Data) ---
csat_score_clean
No Survey    20670
5.0           1899
6.0           1865
3.0           1575
4.0           1526
7.0           1314
8.0           1266
9.0           1092
1.0            595
2.0            571
10.0           568
Name: count, dtype: int64


In [29]:
# This automatically ignores the nulls and gives you the true performance average
avg_csat = df['csat_score'].mean()
#avg_csat
print(f"Average Customer Satisfaction Score: {avg_csat:.2f}")

Average Customer Satisfaction Score: 5.55


In [36]:
# Convert call_timestamp to an actual datetime format
df['call_timestamp'] = pd.to_datetime(df['call_timestamp'])

min=df['call_timestamp'].min()
max=df['call_timestamp'].max()
print(f"{min} to {max}")


2020-10-01 00:00:00 to 2020-10-31 00:00:00


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32941 entries, 0 to 32940
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id                        32941 non-null  object        
 1   customer_name             32941 non-null  object        
 2   sentiment                 32941 non-null  object        
 3   csat_score                12271 non-null  float64       
 4   call_timestamp            32941 non-null  datetime64[ns]
 5   reason                    32941 non-null  object        
 6   city                      32941 non-null  object        
 7   state                     32941 non-null  object        
 8   channel                   32941 non-null  object        
 9   response_time             32941 non-null  object        
 10  call duration in minutes  32941 non-null  int64         
 11  call_center               32941 non-null  object        
 12  csat_score_clean  

In [41]:
# Group by the person handling the call to find their average score and survey count
agent_perf = df.groupby('call_center').agg(
    Avg_CSAT=('csat_score', 'mean'),
    Total_Surveys=('csat_score', 'count')
).reset_index()

# Filter for agents who have a below-average score (less than 5.55) 
# and have received at least a few surveys
low_perf_agents = agent_perf[(agent_perf['Avg_CSAT'] < 5.55) & (agent_perf['Total_Surveys'] > 5)]

# Sort by lowest score first
print("--- Agents Needing Targeted Training ---")
print(low_perf_agents.sort_values(by='Avg_CSAT').head(10))

--- Agents Needing Targeted Training ---
  call_center  Avg_CSAT  Total_Surveys
1  Chicago/IL  5.479438           1994


In [42]:
# 1. Filter the dataset down to just the Chicago call center
chicago_df = df[df['call_center'] == 'Chicago/IL']

# 2. Find which call reasons have the lowest CSAT inside Chicago
chicago_reasons = chicago_df.groupby('reason').agg(
    Avg_CSAT=('csat_score', 'mean'),
    Avg_Duration=('call duration in minutes', 'mean'),
    Total_Calls=('id', 'count')
).reset_index()

print("--- Chicago Performance Breakdown by Reason ---")
print(chicago_reasons.sort_values(by='Avg_CSAT'))

--- Chicago Performance Breakdown by Reason ---
             reason  Avg_CSAT  Avg_Duration  Total_Calls
2    Service Outage  5.334448     25.429870          770
0  Billing Question  5.481299     24.946822         3855
1          Payments  5.625899     25.268262          794


In [43]:
# Group by channel inside Chicago to see performance
chicago_channels = chicago_df.groupby('channel').agg(
    Avg_CSAT=('csat_score', 'mean'),
    Avg_Duration=('call duration in minutes', 'mean'),
    Total_Calls=('id', 'count')
).reset_index()

print("--- Chicago Performance Breakdown by Channel ---")
print(chicago_channels.sort_values(by='Avg_CSAT'))

--- Chicago Performance Breakdown by Channel ---
       channel  Avg_CSAT  Avg_Duration  Total_Calls
2        Email  5.183721     25.375614         1222
1      Chatbot  5.390385     24.880411         1363
3          Web  5.503876     24.520873         1054
0  Call-Center  5.729072     25.307865         1780


In [44]:
df.to_csv('Call_Center_Cleaned.csv', index=False)